# 🗂️ Notebook 2: Amazon Lambda (Serverless) — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/amazon-lambda
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Entities

- **Function** — user code + config.
- **Invocation** — one execution.
- **Container** — sandboxed runtime instance.

## Pydantic models

We use `pydantic` for data validation — it forces us to think about types, required fields, and invariants up front.

In [ ]:
from datetime import datetime
from typing import Literal
from typing import Optional
from pydantic import BaseModel, Field

class Function(BaseModel):
    name: str
    runtime: Literal["python3.11","node20","go1.21"]
    handler: str             # "module.handler"
    memory_mb: int = Field(ge=128, le=10240)
    timeout_s: int = 30

class Invocation(BaseModel):
    id: str
    function: str
    started_at: datetime
    duration_ms: Optional[int] = None
    cold_start: bool = False
    status: Literal["ok","error","timeout"] = "ok"

## HTTP APIs

| Method | Path | What |
|---|---|---|
| PUT | `/functions/{name}` | Create/update function |
| POST | `/functions/{name}/invoke` | Invoke synchronously |
| POST | `/functions/{name}/event` | Async invoke |
| GET | `/invocations/{id}` | Invocation detail |


## Quick demo

In [ ]:
# Conceptual invocation lifecycle
steps = [
    "1. receive event",
    "2. look up function config",
    "3. find warm container? → if yes: dispatch",
    "4. else: allocate slot, clone base VM, init runtime",
    "5. run handler with event payload",
    "6. capture logs/metrics, return response",
    "7. keep container warm for a while",
]
for s in steps: print(s)

## Takeaways

- Small, typed models make the service boundary crisp.
- Public APIs hide internal IDs and expose human-friendly resources.
- Write one happy-path test per endpoint before scaling out.